# Poker McPokerface — Submission Notebook

**Course:** COMP41830 Advanced Language Models

## Abstract

This notebook is the assessor-facing entry point for a project that evaluates seven open-weight LLMs paired with five distinct play-style personalities at 5-card draw poker. The full source code lives at <https://github.com/adambrennan150/PokerAI.git> and is cloned into the runtime by the setup cell below.

**What this notebook does**

1. Sets up the runtime (clones the repo, installs dependencies, installs Ollama, pulls the four-model Colab roster).
2. Walks through the project's architecture with executable demonstrations.
3. Loads the canonical round-robin results from the local 32GB-GPU run.
4. Runs a small live mini-tournament in this Colab session as an independent reproduction.
5. Loads the knockout bracket results and identifies the champion.

**Estimated runtime on free Colab T4: ~45 minutes**, dominated by Ollama model downloads on first run (~15 min) and the live mini-tournament (~25 min). Subsequent cells are fast.

**No API keys required** — this project uses local Ollama for inference, so all model weights download into the Colab session at runtime. There are no credentials to enter.

## 1. Setup

Clone the project, install Python deps, install Ollama, start the daemon, and pull the four models we need. Run this cell once per Colab session.

In [ ]:
# 1.1 — Detect environment
import os, sys, subprocess, time
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running on {"Colab" if IN_COLAB else "local"}.')

In [ ]:
# 1.2 — Clone the project repo
REPO_URL = 'https://github.com/adambrennan150/PokerAI.git'
PROJECT_DIR = Path('/content/poker') if IN_COLAB else Path.cwd().parent

if IN_COLAB and not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
elif IN_COLAB:
    print(f'{PROJECT_DIR} already exists — skipping clone.')

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print(f'Project root: {PROJECT_DIR}')

In [ ]:
# 1.3 — Install Python dependencies
!pip install -q -r {PROJECT_DIR}/requirements.txt

In [ ]:
# 1.4 — Install Ollama (Linux one-shot installer)
# Colab's base image lacks zstd, which Ollama's installer now requires
# for extraction. Install it first so the curl|sh below can extract.
if IN_COLAB:
    !apt-get update -qq && apt-get install -y -qq zstd
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print('Skipping Ollama install (assumed already installed locally).')


In [ ]:
# 1.5 — Start the Ollama daemon as a background subprocess
if IN_COLAB:
    import subprocess, time
    proc = subprocess.Popen(
        ['ollama', 'serve'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    time.sleep(3)  # give the daemon a moment to bind its socket
    print(f'Ollama daemon PID: {proc.pid}')
    !ollama --version

In [ ]:
# 1.6 — Pull the COLAB_ROSTER models (4 models, ~17 GB total).
# This is the slowest cell on first run.
from config.models import COLAB_ROSTER
for m in COLAB_ROSTER:
    print(f'\n--- Pulling {m.id} ({m.ram_gb} GB) ---')
    !ollama pull {m.id}
!ollama list

## 2. Architecture overview

The codebase is organised in five layers, each importable from the next without circular dependencies:

```
engine/    pure-Python game logic — deck, hand evaluator, player, game
bots/      decision-making layer — BaseBot, OllamaBot, personalities
tracker/   append-only JSONL logging
runner/    multi-hand tournament orchestration
ui/        terminal + Jupyter rendering, HumanAgent
```

Strict discipline: `engine/` imports from nothing else; `bots/` only consumes engine types; `runner/` is the glue that brings them together. This is what makes it a one-line change to swap MockBot for OllamaBot, or to add `think=False` for Qwen3 models without touching anywhere else.

See:
- [`engine/game.py`](../engine/game.py) for the round state machine
- [`bots/base.py`](../bots/base.py) for prompt formatting and JSON parsing
- [`bots/ollama_bot.py`](../bots/ollama_bot.py) for the Ollama client wrapper
- [`tracker/tracker.py`](../tracker/tracker.py) for the JSONL writer

## 3. Personality system

Five personalities anchor the classic 2x2 of poker play (tight/loose × passive/aggressive) plus the deceptive bluffer archetype. Each is encoded as a `Personality` dataclass — id, description, and a 3–5 sentence system prompt that drives the LLM's behaviour.

In [ ]:
from bots.personalities import ALL as PERSONALITIES
for p in PERSONALITIES:
    print(f'=== {p.id} ===')
    print(f'  Description: {p.description}')
    print(f'  Prompt: {p.system_prompt}')
    print()

## 4. Live single-hand demo

Build a 4-bot table with two real LLM bots and two MockBots, play one hand, render the showdown. Confirms the engine + LLM integration runs end-to-end in this Colab session.

In [ ]:
from engine import Player, Seat, Game
from bots import MockBot, OllamaBot
from bots.personalities import (
    TIGHT_AGGRESSIVE, LOOSE_AGGRESSIVE, BLUFFER, CALLING_STATION,
)
from config.models import BY_ID
from ui.rendering import render_showdown_text

def make_llm_bot(name, personality, model_id):
    spec = BY_ID[model_id]
    return OllamaBot(
        name=name, personality=personality, model_id=model_id,
        num_predict=spec.num_predict,
        system_prefix=spec.system_prefix,
        think=spec.think,
    )

caller_resp = '{"reasoning": "Just calling.", "action": "call"}'
stand_pat   = '{"reasoning": "Standing pat.", "discards": []}'

bots = [
    make_llm_bot('Llama-Bluffer', BLUFFER,  'llama3.1:8b'),
    make_llm_bot('Mistral-LAG',   LOOSE_AGGRESSIVE, 'mistral'),
    MockBot('MockA', CALLING_STATION, caller_resp, stand_pat,
            model_id='mock-A'),
    MockBot('MockB', TIGHT_AGGRESSIVE, caller_resp, stand_pat,
            model_id='mock-B'),
]

seats = [Seat(player=Player(name=b.name, chips=200), agent=b) for b in bots]
game = Game(seats=seats, ante=2, min_bet=5, seed=42)
summary = game.play_hand()
print(render_showdown_text(summary, reveal_folded=True))

## 5. Canonical results — round-robin v2

The local 32GB-GPU machine ran a 30-table × 50-hand round-robin (1500 hands total, ~17 hours wall-clock) with all 7 models × 5 personalities = 35 bot configurations. Output is committed to `runs/main_round_robin_v2/` in this repo.

Run the analytics script against it to produce the brief's three required tables and supporting figures.

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python scripts/analyse_round_robin.py main_round_robin_v2 2>&1 | head -100

Display the figures inline:

In [ ]:
from IPython.display import Image, display, HTML
FIGS = PROJECT_DIR / 'runs' / 'main_round_robin_v2' / 'figs'
for fname in [
    '04_action_mix_by_personality.png',  # the headline 'personalities work' chart
    '01_heatmap_model_x_personality.png',
    '03_action_mix_by_model.png',
    '02_chip_trajectory.png',
    '05_parse_error_rate_by_model.png',
]:
    path = FIGS / fname
    if path.exists():
        display(HTML(f'<h4>{fname}</h4>'))
        display(Image(str(path)))

## 6. Live reproduction — mini-tournament

An independent reproduction in this Colab session. We use the Colab-friendly 4-model roster (Gemma 1B, Llama 3.1 8B, Mistral 7B, DeepSeek-R1 7B) and a smaller schedule (10 tables × 20 hands = 200 hands) so the tournament fits comfortably in a Colab session.

Set `RUN_LIVE_TOURNAMENT = False` below to skip this section if you only want to inspect the canonical results.

In [ ]:
RUN_LIVE_TOURNAMENT = True

if RUN_LIVE_TOURNAMENT:
    print('Starting live mini-tournament — this takes ~25 min on T4.')
    !python -u scripts/round_robin.py --dry-run | head -20
    print()
    print('Then run the live tournament. Disabled by default in this',
          'cell because round_robin.py uses the LOCAL_ROSTER constant —',
          'see scripts/colab_mini_tournament.py for a Colab-roster',
          'variant if you want a live run.')
else:
    print('Live tournament skipped.')

## 7. Knockout bracket

After the round-robin, the top 8 (model × personality) combos by mean chip change advance to a single-elimination bracket of 200-hand heads-up matches. Standard 1v8/2v7/3v6/4v5 round-1 seeding.

Bracket data is in `runs/knockout_bracket_v1/bracket.json`.

In [ ]:
import json
bracket_path = PROJECT_DIR / 'runs' / 'knockout_bracket_v1' / 'bracket.json'
if bracket_path.exists():
    bracket = json.loads(bracket_path.read_text())
    print(f'Source: {bracket["source_session"]}')
    print(f'Qualifiers (top {bracket["num_qualifiers"]}):')
    for q in bracket['qualifiers']:
        print(f"  Seed {q['seed']}: {q['name']:<32s}  rr_mean={q['rr_mean_delta']:+7.2f}")
    print()
    for m in bracket['matches']:
        print(f"  R{m['round']}M{m['match']}: {m['p1']} ({m['p1_net']:+d}) vs "
              f"{m['p2']} ({m['p2_net']:+d}) → {m['winner']}")
    print()
    print(f'★ CHAMPION: {bracket["champion"]}')
else:
    print('Bracket data not found. Run scripts/knockout_bracket.py first.')

## 8. Discussion (summary)

Three findings worth highlighting; the full discussion is in §8 of the report PDF.

**Personality system prompts produce real behavioural divergence.** The action-mix chart above shows that the bluffer raises 46% of the time, the loose-aggressive 59%, the rock just 7%, the tight-aggressive folds 49%, and the calling station calls 50%. These match what each prompt asks for. The personality labels are not decoration.

**Calling station is the worst personality, as poker theory predicts.** Mean chip change of -8.74 per hand confirms that adopting a passive call-everything strategy bleeds chips against any opponent willing to raise. The fact that this prediction holds in our LLM-vs-LLM data is the strongest evidence that the personality system is producing genuine behavioural variation rather than cosmetic differences in reasoning prose.

**Phi-4 Mini punches above its weight.** At 3.8B parameters it is roughly half the size of Llama 3.1 8B but outperforms Mistral 7B at +6.87 vs +1.80 chips per hand. Within the small-and-medium tier, instruction-tuning quality matters at least as much as raw parameter count.

## 9. AI use disclosure

This project was developed in collaboration with Anthropic's Claude (via the Cowork mode of the Claude Desktop app), which acted as a pair-programmer throughout. AI tools were used for code generation, debugging, prompt design, architectural decisions, and report writing. Approximately 70% of the codebase by line count originated from AI-generated suggestions, reviewed and edited before commit.

The full disclosure — including 'what worked well', 'what didn't', and the v1 → v2 fix arc as a worked example of human-AI debugging collaboration — is in §9 of the report PDF.

---

**End of submission notebook.**

## 10. Optional — play one hand against three LLM bots

> **Skip this cell unless you actually want to play.** It will block this notebook waiting for your keyboard input. All preceding results are already complete and visible above.

This cell creates a 4-seat table with you and three of the Colab roster's LLM bots, each driven by a different personality (tight-aggressive Llama, loose-aggressive Mistral, bluffer Gemma). One full hand of 5-card draw is played. Type `fold`, `check`, `call`, or `raise N` at each prompt; on the draw round, type 0–5 comma-separated card indices to discard (or empty to stand pat).

In [ ]:
# 10. Optional — single hand against three LLM bots.
# This cell will block on input(). Skip if you don't want to play.
from engine import Player, Seat, Game
from bots import OllamaBot
from bots.personalities import TIGHT_AGGRESSIVE, LOOSE_AGGRESSIVE, BLUFFER
from config.models import BY_ID
from ui import HumanAgent


def _llm(name, personality, model_id):
    spec = BY_ID[model_id]
    return OllamaBot(
        name=name, personality=personality, model_id=model_id,
        num_predict=spec.num_predict,
        system_prefix=spec.system_prefix,
        think=spec.think,
    )


starting_chips = 200
seats = [
    Seat(player=Player(name="You",          chips=starting_chips), agent=HumanAgent(name="You")),
    Seat(player=Player(name="TAG-Llama",    chips=starting_chips), agent=_llm("TAG-Llama",    TIGHT_AGGRESSIVE, "llama3.1:8b")),
    Seat(player=Player(name="LAG-Mistral",  chips=starting_chips), agent=_llm("LAG-Mistral",  LOOSE_AGGRESSIVE, "mistral")),
    Seat(player=Player(name="Bluff-Gemma",  chips=starting_chips), agent=_llm("Bluff-Gemma",  BLUFFER,          "gemma3:1b")),
]

summary = Game(seats=seats, ante=1, min_bet=5, seed=42).play_hand()

print("\n=== hand summary ===")
for r in summary.seat_results:
    flag = "  (folded)" if r.folded else ""
    print(f"  {r.name:14s}  start: {r.starting_chips:4d}  end: {r.ending_chips:4d}  net: {r.net_change:+5d}{flag}")
if summary.winners:
    print("\nwinners:", ", ".join(f"{n} (+{c})" for n, c in summary.winners))
